# PUNTO 3 — Academic Season Analysis
### University Academic Database (AD) — Oracle Live SQL
**Analysis of previous academic years, course information, department info, and faculty info.**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from io import StringIO

# ── Color palette ──────────────────────────────────────────────
COLORS = ['#2D6A9F', '#E8A838', '#2ECC71', '#E74C3C', '#9B59B6', '#1ABC9C']
plt.rcParams['figure.facecolor'] = '#F7F9FC'
plt.rcParams['axes.facecolor']   = '#FFFFFF'
plt.rcParams['font.family']      = 'DejaVu Sans'
plt.rcParams['axes.spines.top']  = False
plt.rcParams['axes.spines.right']= False

print('Libraries loaded ✓')

In [ ]:
# ── Load data ──────────────────────────────────────────────────
consolidated_csv = """SESSION_ID,SESSION_NAME,DEPARTMENT_ID,DEPARTMENT_NAME,HOD,COURSE_ID,COURSE_NAME,FACULTY_ID,FACULTY_NAME,JOB_ID,HIRE_DATE,SALARY
100,SPRING SESSION,10,ACCOUNTING,MARK SMITH,192,COST ACCOUNTING,100,Steven King,FA_ST,2013-06-17,4000
100,SPRING SESSION,10,ACCOUNTING,MARK SMITH,193,STRATEGIC TAX PLANNING FOR BUSINESS,100,Steven King,FA_ST,2013-06-17,4000
100,SPRING SESSION,10,ACCOUNTING,MARK SMITH,190,PRINCIPLES OF ACCOUNTING,104,Bruce Ernst,FA_SF,2012-05-21,6000
100,SPRING SESSION,10,ACCOUNTING,MARK SMITH,191,INTRODUCTION TO BUSINESS LAW,105,David Austin,FA_SF,2015-06-25,4800
100,SPRING SESSION,40,LITERATURE,ANITA TAYLOR,189,COLLEGE READING,101,Neena Kochhar,FA_ST,2015-09-21,6000
200,FALL SESSION,20,BIOLOGY,DAVE GOLD,196,INTRODUCTION TO PLANT PHYSIOLOGY,102,Lex De Haan,FA_AF,2011-01-13,15000
200,FALL SESSION,20,BIOLOGY,DAVE GOLD,194,GENERAL BIOLOGY,103,Alexander Hunold,FA_AF,2014-01-03,9000
200,FALL SESSION,20,BIOLOGY,DAVE GOLD,195,CELL BIOLOGY,104,Bruce Ernst,FA_SF,2012-05-21,6000
200,FALL SESSION,20,BIOLOGY,DAVE GOLD,197,MARINE BIOLOGY,109,Daniel Faviet,FA_HOD,2012-08-16,39000
200,FALL SESSION,40,LITERATURE,ANITA TAYLOR,176,BUSINESS WRITING,101,Neena Kochhar,FA_ST,2015-09-21,6000
300,SUMMER SESSION,30,COMPUTER SCIENCE,LINDA BROWN,199,WEB PROGRAMMING,106,Valli Pataballa,FA_PF,2011-02-05,28000
300,SUMMER SESSION,30,COMPUTER SCIENCE,LINDA BROWN,198,SIMULATION AND MODELING,106,Valli Pataballa,FA_PF,2011-02-05,28000
300,SUMMER SESSION,30,COMPUTER SCIENCE,LINDA BROWN,187,DATA STRUCTURES,107,Diana Lorentz,FA_PF,2010-02-07,18000
300,SUMMER SESSION,30,COMPUTER SCIENCE,LINDA BROWN,188,OOAD,108,Nancy Greenberg,FA_HOD,2012-08-17,21200
300,SUMMER SESSION,40,LITERATURE,ANITA TAYLOR,175,AMERICAN LITERATURE,107,Diana Lorentz,FA_PF,2010-02-07,18000
"""

dept_csv = """DEPARTMENT_NAME,HOD,TOTAL_CURSOS
LITERATURE,ANITA TAYLOR,3
COMPUTER SCIENCE,LINDA BROWN,4
ACCOUNTING,MARK SMITH,4
BIOLOGY,DAVE GOLD,4
"""

faculty_csv = """FACULTY_NAME,JOB_ID,CURSOS_ASIGNADOS
Nancy Greenberg,FA_HOD,1
Alexander Hunold,FA_AF,1
Lex De Haan,FA_AF,1
David Austin,FA_SF,1
Daniel Faviet,FA_HOD,1
Diana Lorentz,FA_PF,2
Valli Pataballa,FA_PF,2
Neena Kochhar,FA_ST,2
Steven King,FA_ST,2
Bruce Ernst,FA_SF,2
"""

df   = pd.read_csv(StringIO(consolidated_csv))
dept = pd.read_csv(StringIO(dept_csv))
fac  = pd.read_csv(StringIO(faculty_csv))

df['HIRE_DATE'] = pd.to_datetime(df['HIRE_DATE'])
df['HIRE_YEAR'] = df['HIRE_DATE'].dt.year

print(f'Consolidated rows: {len(df)}')
print(f'Departments: {df["DEPARTMENT_NAME"].nunique()}')
print(f'Sessions: {df["SESSION_NAME"].nunique()}')
print(f'Faculty members: {df["FACULTY_NAME"].nunique()}')
df.head()

---
## GRAPH 1 — Courses per Department
**Question:** Which department offers the most courses? Which needs strengthening?

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

dept_sorted = dept.sort_values('TOTAL_CURSOS', ascending=True)
colors = ['#E74C3C' if v < 4 else '#2D6A9F' for v in dept_sorted['TOTAL_CURSOS']]

bars = ax.barh(dept_sorted['DEPARTMENT_NAME'], dept_sorted['TOTAL_CURSOS'],
               color=colors, edgecolor='white', height=0.55)

for bar, val in zip(bars, dept_sorted['TOTAL_CURSOS']):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            f'{val} courses', va='center', fontsize=11, fontweight='bold')

ax.set_xlim(0, 5.5)
ax.set_xlabel('Number of Courses', fontsize=11)
ax.set_title('Courses per Department', fontsize=14, fontweight='bold', pad=15)

red_patch  = mpatches.Patch(color='#E74C3C', label='Needs strengthening (< 4 courses)')
blue_patch = mpatches.Patch(color='#2D6A9F', label='Adequate (4 courses)')
ax.legend(handles=[red_patch, blue_patch], loc='lower right', fontsize=9)

ax.axvline(x=4, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='Average')
plt.tight_layout()
plt.savefig('graph1_courses_per_department.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n📌 FINDING: Literature only has 3 courses — the weakest department.')

---
## GRAPH 2 — Courses per Academic Session
**Question:** How are courses distributed across sessions?

In [ ]:
session_counts = df.groupby('SESSION_NAME')['COURSE_ID'].nunique().reset_index()
session_counts.columns = ['SESSION', 'COURSES']
session_order = ['SPRING SESSION', 'FALL SESSION', 'SUMMER SESSION']
session_counts['SESSION'] = pd.Categorical(session_counts['SESSION'], categories=session_order, ordered=True)
session_counts = session_counts.sort_values('SESSION')

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(session_counts['SESSION'], session_counts['COURSES'],
              color=['#2D6A9F', '#E8A838', '#2ECC71'], edgecolor='white', width=0.5)

for bar, val in zip(bars, session_counts['COURSES']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            str(val), ha='center', fontsize=13, fontweight='bold')

ax.set_ylim(0, 8)
ax.set_ylabel('Number of Unique Courses', fontsize=11)
ax.set_title('Courses per Academic Session', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('graph2_courses_per_session.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n📌 FINDING: Summer Session has the highest course load (5 courses). Spring and Fall are balanced at 5 each.')

---
## GRAPH 3 — Teaching Load per Faculty Member
**Question:** Who is underloaded? Which roles need more faculty?

In [ ]:
fac_sorted = fac.sort_values('CURSOS_ASIGNADOS', ascending=True)
job_colors = {
    'FA_HOD': '#E74C3C', 'FA_AF': '#E8A838',
    'FA_SF':  '#2ECC71', 'FA_PF': '#9B59B6', 'FA_ST': '#2D6A9F'
}
bar_colors = [job_colors[j] for j in fac_sorted['JOB_ID']]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(fac_sorted['FACULTY_NAME'], fac_sorted['CURSOS_ASIGNADOS'],
               color=bar_colors, edgecolor='white', height=0.6)

for bar, val, job in zip(bars, fac_sorted['CURSOS_ASIGNADOS'], fac_sorted['JOB_ID']):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            f'{val}  [{job}]', va='center', fontsize=10)

ax.set_xlim(0, 3.2)
ax.axvline(x=1.5, color='gray', linestyle='--', linewidth=1, alpha=0.6, label='Average (1.5)')
ax.set_xlabel('Courses Assigned', fontsize=11)
ax.set_title('Teaching Load per Faculty Member', fontsize=14, fontweight='bold', pad=15)

legend_patches = [mpatches.Patch(color=v, label=k) for k, v in job_colors.items()]
ax.legend(handles=legend_patches, title='Job Role', fontsize=9, loc='lower right')
plt.tight_layout()
plt.savefig('graph3_teaching_load.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n📌 FINDING: FA_HOD and FA_AF roles carry the lightest teaching load (1 course each). FA_PF, FA_ST, FA_SF have heavier loads.')

---
## GRAPH 4 — Department Distribution per Session (Stacked Bar)
**Question:** Which departments are active in each session?

In [ ]:
pivot = df.groupby(['SESSION_NAME', 'DEPARTMENT_NAME'])['COURSE_ID'].nunique().unstack(fill_value=0)
pivot = pivot.reindex(['SPRING SESSION', 'FALL SESSION', 'SUMMER SESSION'])

dept_colors = {'ACCOUNTING':'#2D6A9F','BIOLOGY':'#2ECC71','COMPUTER SCIENCE':'#9B59B6','LITERATURE':'#E8A838'}
colors_list = [dept_colors[c] for c in pivot.columns]

fig, ax = plt.subplots(figsize=(9, 5))
pivot.plot(kind='bar', stacked=True, ax=ax, color=colors_list, edgecolor='white', width=0.5)

ax.set_xticklabels(pivot.index, rotation=0, fontsize=11)
ax.set_ylabel('Number of Courses', fontsize=11)
ax.set_title('Department Activity per Academic Session', fontsize=14, fontweight='bold', pad=15)
ax.legend(title='Department', fontsize=9, bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.savefig('graph4_dept_per_session.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n📌 FINDING: Literature appears in ALL 3 sessions but with only 1 course each time. Other departments are concentrated in specific sessions.')

---
## GRAPH 5 — Salary Distribution by Job Role
**Question:** Are salary levels aligned with responsibilities?

In [ ]:
salary_by_role = df.drop_duplicates('FACULTY_ID').groupby('JOB_ID')['SALARY'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
colors_salary = [job_colors.get(j, '#999') for j in salary_by_role.index]
bars = ax.bar(salary_by_role.index, salary_by_role.values,
              color=colors_salary, edgecolor='white', width=0.5)

for bar, val in zip(bars, salary_by_role.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 300,
            f'${val:,.0f}', ha='center', fontsize=11, fontweight='bold')

ax.set_ylabel('Average Salary (USD)', fontsize=11)
ax.set_title('Average Salary by Job Role', fontsize=14, fontweight='bold', pad=15)
ax.set_ylim(0, max(salary_by_role.values) * 1.2)
plt.tight_layout()
plt.savefig('graph5_salary_by_role.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n📌 FINDING: FA_HOD earns significantly more ($30,100 avg) despite having the lowest teaching load — consistent with a management/leadership role.')

---
## GRAPH 6 — Faculty Hiring Timeline
**Question:** When was faculty hired? Are there gaps in hiring?

In [ ]:
hire_data = df.drop_duplicates('FACULTY_ID')[['FACULTY_NAME','HIRE_DATE','JOB_ID','SALARY']].sort_values('HIRE_DATE')

fig, ax = plt.subplots(figsize=(11, 5))
for i, row in hire_data.iterrows():
    color = job_colors.get(row['JOB_ID'], '#999')
    ax.scatter(row['HIRE_DATE'], row['SALARY'], color=color, s=120, zorder=5)
    ax.text(row['HIRE_DATE'], row['SALARY'] + 700, row['FACULTY_NAME'].split()[-1],
            ha='center', fontsize=8, color='#333')

ax.set_xlabel('Hire Date', fontsize=11)
ax.set_ylabel('Salary (USD)', fontsize=11)
ax.set_title('Faculty Hiring Timeline vs Salary', fontsize=14, fontweight='bold', pad=15)
legend_patches = [mpatches.Patch(color=v, label=k) for k, v in job_colors.items()]
ax.legend(handles=legend_patches, title='Job Role', fontsize=9)
plt.tight_layout()
plt.savefig('graph6_hiring_timeline.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n📌 FINDING: Most faculty were hired between 2010-2015. No new hires after 2015 — the university may need fresh faculty recruitment before the new academic season.')

---
## GRAPH 7 — NULL / Duplicate / Data Quality Summary
**Question:** How complete is the data?

In [ ]:
categories  = ['NULL Values', 'Duplicate Courses', 'Duplicate Depts', 'Duplicate Sessions', 'Faculty w/o Courses']
values      = [0, 0, 0, 0, 0]
bar_colors2 = ['#2ECC71' if v == 0 else '#E74C3C' for v in values]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(categories, values, color=bar_colors2, edgecolor='white', width=0.5)

for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, 0.05,
            '✓ NONE', ha='center', fontsize=12, fontweight='bold', color='white')

ax.set_ylim(0, 3)
ax.set_ylabel('Issues Found', fontsize=11)
ax.set_title('Data Quality Check — Consolidated Table', fontsize=14, fontweight='bold', pad=15)
ax.set_xticklabels(categories, rotation=15, ha='right', fontsize=10)
plt.tight_layout()
plt.savefig('graph7_data_quality.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n📌 FINDING: The database is clean — zero NULLs, zero duplicates. Data integrity is strong.')

---
## SUMMARY — Analysis, Proposals & Conclusions

In [ ]:
summary = """
╔══════════════════════════════════════════════════════════════════════╗
║         PUNTO 3 — ACADEMIC SEASON ANALYSIS SUMMARY                  ║
╠══════════════════════════════════════════════════════════════════════╣
║  FINDINGS:                                                           ║
║  1. Data Quality: EXCELLENT — 0 NULLs, 0 duplicates                 ║
║  2. Weakest Department: LITERATURE (only 3 courses vs 4 avg)         ║
║  3. Underloaded Roles: FA_HOD & FA_AF (1 course per faculty)         ║
║  4. No new faculty hired after 2015 — aging faculty base             ║
║  5. Literature appears in all 3 sessions with only 1 course each     ║
║  6. HOD role earns most but teaches least → management overhead      ║
╠══════════════════════════════════════════════════════════════════════╣
║  PROPOSALS FOR THE NEW ACADEMIC SEASON:                              ║
║  → Expand Literature dept: add at least 1-2 new courses              ║
║  → Recruit new faculty (no hires since 2015)                         ║
║  → Redistribute load from FA_HOD to teaching roles                   ║
║  → Assign a second course to FA_AF faculty (Lex De Haan, Hunold)     ║
║  → Review if FA_SF (David Austin) can handle a second course         ║
╠══════════════════════════════════════════════════════════════════════╣
║  CONCLUSIONS:                                                        ║
║  The university's data is clean and well-structured. The main        ║
║  academic gap is in the Literature department and the imbalanced     ║
║  teaching load across roles. Strategic hiring and course expansion   ║
║  in Literature would significantly strengthen academic readiness     ║
║  for the upcoming season.                                            ║
╚══════════════════════════════════════════════════════════════════════╝
"""
print(summary)